<a href="https://colab.research.google.com/github/gav-ip/ML-zero/blob/main/transformer_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import jax as jnn
from jax import random
import jax.numpy as jnp
import flax.nnx as nnx 
import numpy as np
import random as py_random
import optax as opt

In [3]:
USE_GPU = False
device = jnn.devices("cpu")[0] if not USE_GPU else jnn.devices()[0]
n_examples = 100000
block_size = 12
batch_size = 256
eval_iters = 200
n_embed = 32
n_heads = 4
n_blocks = 4
dropout = 0.2

print(device)

cpu:0


In [4]:
vocab = {0:'0', 1:'1', 2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8', 9:'9', 10:'+', 11:'=', 12:' '}
vocab_size = len(vocab)

In [5]:
stoi = {vocab[i]:i for i, ch in enumerate(vocab)}
itos = {i:vocab[i] for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[int(i)] for i in l])

print(encode("8+2 =10 "))
print(decode(encode("8+7 =15 ")))

[8, 10, 2, 12, 11, 1, 0, 12]
8+7 =15 


In [6]:
def build_key(current_key, min_val, max_val):
  # Split the key to generate 'a' and 'b', and get a new key for the next iteration
  subkey_a, subkey_b, next_key = random.split(current_key, 3)
  # Generate a single integer value using .item()
  a = random.randint(subkey_a, shape=(), minval=min_val, maxval=max_val).item()
  b = random.randint(subkey_b, shape=(), minval=min_val, maxval=max_val).item()
  return a, b, next_key

In [7]:
def cateogrize_sum(current_key, split, min_sum_val, max_sum_val):
  target_digit_count = int(n_examples * split)

  collected_examples = []

  while (len(collected_examples) < target_digit_count):
    # Call build_key to get a single (a, b) pair and an updated key
    a, b, current_key = build_key(current_key, min_sum_val, max_sum_val) # Always generate a and b between 0-999

    s = a + b # s is now a scalar integer

    # Check if the sum 's' falls into the desired range (one-digit, two-digit, or three-digit)
    # and ensure sum s is not too large (max 3 digits for 0-999)
    if min_sum_val <= s <= max_sum_val:
      c_reversed = str(s)[::-1] # Now 's' is scalar, str(s) works as expected
      key_str = f"{a:>3}+{b:>3}={c_reversed:<4}"
      collected_examples.append(key_str)

  return collected_examples, current_key # Return the list of strings and the updated key

In [8]:
def build_data():
  initial_key = random.PRNGKey(0) # Initialize JAX random key once

  one_digit_sums = []
  two_digit_sums = []
  three_digit_sums = [] # Renamed from 'other_digit_sums' for clarity

  # Pass and update the JAX random key correctly for each categorization call
  # The min_sum_val and max_sum_val here define the range for the *sum's digits*
  one_digit_sums, initial_key = cateogrize_sum(initial_key, 0.25, 0, 9)
  two_digit_sums, initial_key = cateogrize_sum(initial_key, 0.25, 10, 99)
  three_digit_sums, initial_key = cateogrize_sum(initial_key, 0.5, 100, 999)

  # Combine all generated examples
  all_examples = one_digit_sums + two_digit_sums + three_digit_sums

  # Shuffle the combined list to mix the categories randomly
  py_random.shuffle(all_examples)

  text = all_examples # 'text' is now a list of formatted strings

  return text

# Call build_data and unpack the results
text = build_data()

In [9]:
encoded_text = [encode(t) for t in text]
# Convert the list of encoded lists into a JAX 2D array
data = jnp.array(encoded_text, device=device)

print(f"Data shape: {data.shape}")
data

Data shape: (100000, 12)


Array([[12,  6,  6, ...,  9, 12, 12],
       [12, 12,  2, ..., 12, 12, 12],
       [ 2,  6,  6, ...,  9,  5, 12],
       ...,
       [ 4,  4,  8, ...,  7,  7, 12],
       [12, 12,  8, ..., 12, 12, 12],
       [ 3,  4,  3, ...,  8,  9, 12]], dtype=int32)

In [10]:
print(decode(data[1]))

  2+  7=9   


In [11]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [12]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = jnp.randint(0, len(data) - block_size, (batch_size,))
    x = jnp.stack([data[i:i+block_size] for i in ix])
    y = jnp.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


In [13]:
# call jax.lax.stop_gradient for no_grad
# def estimate_loss():
#     out = {}
#     model.eval()
#     for split in ['train', 'val']:
#         losses = jnp.zeros(eval_iters)
#         for k in range(eval_iters):
#             X, Y = get_batch(split)
#             logits, loss = model(X, Y)
#             losses[k] = loss.item()
#         out[split] = losses.mean()
#     model.train()
#     return out

In [17]:
import flax.nnx as nnx

rngs = nnx.Rngs(0)
class MultiHeadAttention(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.key = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.query = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.value = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.proj = nnx.Linear(n_embed, n_embed, rngs=rngs)
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

class FeedForward(nnx.Module):
    def __init__(self, x: n_embed, *, rngs: nnx.Rngs):
        self.net = nnx.Sequential(
            nnx.Linear(x, 4 * x, rngs=rngs),
            nnx.gelu(x),
            nnx.Linear(4 * x, x, rngs=rngs),
            nnx.Dropout(dropout, rngs=rngs),
        )

class Block(nnx.Module):
    def __init__(self, n_embd, n_head, *, rngs: nnx.Rngs):
        self.sa = MultiHeadAttention(rngs=rngs)
        self.ff = FeedForward(n_embd, rngs=rngs)
        self.ln1 = nnx.LayerNorm(n_embd, rngs=rngs)
        self.ln2 = nnx.LayerNorm(n_embd, rngs=rngs)

class AdditionModel(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.token_embedding_table = nnx.Embed(vocab_size, n_embed, rngs=rngs)
        self.position_embedding_table = nnx.Embed(block_size, n_embed, rngs=rngs)
        self.blocks = nnx.Sequential(
            *[Block(n_embed, n_heads, rngs=rngs) for _ in range(n_blocks)]
        )
        self.ln_f = nnx.LayerNorm(n_embed, rngs=rngs)
        self.lm_head = nnx.Linear(n_embed, vocab_size, rngs=rngs)

model = AdditionModel(rngs=nnx.Rngs(0))